# ROV Beam Visualizer with Lux Drop-off Bands (v3)
This version includes:
- Sliders for all angle and ROV parameters
- Main beam angle + lux falloff zones
- Labels off to the side and not under the beams
- Beam width calculations shown next to the plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

lux_levels = [("Main", 'deepskyblue'),
              ("100%", 'red'),
              ("90%", 'orange'),
              ("80%", 'gold'),
              ("70%", 'yellow')]

def plot_lux_beams(main_angle, angle_100, angle_90, angle_80, angle_70,
                   show_main, show_100, show_90, show_80, show_70,
                   rov_altitude, light_spacing, light_tilt):

    angles = [main_angle, angle_100, angle_90, angle_80, angle_70]
    shows = [show_main, show_100, show_90, show_80, show_70]
    beam_widths = []

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.set_aspect('equal')
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(0, 1.5)

    def draw_cone(x_center, angle_deg, color, label, tilt_deg):
        half_angle_rad = np.radians(angle_deg / 2)
        beam_radius = np.tan(half_angle_rad) * rov_altitude
        full_width = 2 * beam_radius
        beam_widths.append((label, round((2 * beam_radius + light_spacing), 3)))

        # Convert tilt to radians
        tilt_rad = np.radians(tilt_deg)

        # Cone direction lines
        left_x = x_center + np.sin(-half_angle_rad + tilt_rad) * rov_altitude
        right_x = x_center + np.sin(half_angle_rad + tilt_rad) * rov_altitude
        base_y = 0
        top_y = rov_altitude

        ax.plot([x_center, left_x], [top_y, base_y], color=color, alpha=0.6)
        ax.plot([x_center, right_x], [top_y, base_y], color=color, alpha=0.6)
        ax.fill_between([left_x, right_x], base_y, base_y + 0.01, color=color, alpha=0.2)

    light_positions = [-light_spacing / 2, light_spacing / 2]
    light_tilts = [+light_tilt, -light_tilt]  # outward tilt

    for i, (lux_label, color) in enumerate(lux_levels):
        if shows[i]:
            for x, tilt in zip(light_positions, light_tilts):
                draw_cone(x, angles[i], color, lux_label, tilt)

    # Draw lux labels on the right
    label_x = 2.7
    base_y = 1.35
    for i, (lux_label, color) in enumerate(lux_levels):
        if shows[i]:
            ax.text(label_x, base_y - i * 0.2, f"{lux_label} Lux Band", color=color, fontsize=12, ha='left')

    # Beam widths
    text_x = 2.7
    base_y = 0.4
    for i, (label, width) in enumerate(beam_widths):
        ax.text(text_x, base_y - i * 0.15, f"{label} width: {width} m", fontsize=10, ha='left')

    ax.set_title("ROV Light Beam Spread with Tilted Lights and Lux Drop-Off Zones", fontsize=14)
    ax.set_xlabel("Seafloor (meters)")
    ax.set_ylabel("Height (meters)")
    ax.grid(True)
    plt.show()

# Interact
widgets.interact(
    plot_lux_beams,
    main_angle=widgets.FloatSlider(value=75, min=10, max=135, step=1, description='Main Beam (°)'),
    angle_100=widgets.FloatSlider(value=40, min=10, max=135, step=1, description='100% Lux (°)'),
    angle_90=widgets.FloatSlider(value=50, min=10, max=135, step=1, description='90% Lux (°)'),
    angle_80=widgets.FloatSlider(value=60, min=10, max=135, step=1, description='80% Lux (°)'),
    angle_70=widgets.FloatSlider(value=70, min=10, max=135, step=1, description='70% Lux (°)'),
    show_main=widgets.Checkbox(value=True, description='Show Main'),
    show_100=widgets.Checkbox(value=True, description='Show 100%'),
    show_90=widgets.Checkbox(value=True, description='Show 90%'),
    show_80=widgets.Checkbox(value=True, description='Show 80%'),
    show_70=widgets.Checkbox(value=True, description='Show 70%'),
    rov_altitude=widgets.FloatSlider(value=0.8, min=0.1, max=2.0, step=0.05, description='ROV Altitude (m)'),
    light_spacing=widgets.FloatSlider(value=0.514, min=0.1, max=2.0, step=0.01, description='Light Spacing (m)'),
    light_tilt=widgets.FloatSlider(value=-10, min=-45, max=45, step=1, description='Light Tilt (°)')
)
